In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import linregress
from IPython.display import display, Markdown, Latex
import pandas as pd
from uncertainties import ufloat
from uncertainties import unumpy as unp
from scipy.optimize import curve_fit
from scipy import constants as const 
import math
from matplotlib.patches import Rectangle
from matplotlib.ticker import MaxNLocator

# PW 12
von Moritz Bacher und Emilia Frei
13.1.2026

# PW 12

## Erklärung des ersten Experiments

In diesem Experiment wird das Verhalten eines Drehpendels, einmal ohne einer anregenden Kraft und einmal mit einer anregenden Kraft, beobachtet.
Im Fall einer freien Schwingung, folgt die Bewegung des Pendels folgender Differentialgleichung:
$$
ma + kv + Dx = 0
$$
wobei $k$ die Dämpfungs- oder Reibungskonstante mit SI-EInheit $\frac{kg}{s}$ ist und $D$ eine Systemabhängige Konstante. 
Im Fall einer erzwungenen Schwingung folgt die Bewegung des Pendels folgender Differentialgleichung:
$$
ma + kv + Dx = F_0 \cdot cos(\omega t)
$$

Unter Dämpfung erfährt die Schwingungsfrequenz $\omega_D$ des Pendels eine Abweichung von der Eigenfrequenz $\omega_0$, welcher der folgenden Gleichung folgt:
$$
\omega_D = \sqrt{\omega_0^2 - \delta^2}
$$
wobei $\delta$ den Dämpfungsskoeffizienten beschreibt, welcher mittels eines linearen Fits des Zusammenhangs
$$
\ln{\frac{x_i}{x_0}} = -\delta t_i
$$
als negative Steigung des Fits bestimmt werden kann.
Der Gütefaktor $Q$ wird wie folgt bestimmt:
$$
Q = \frac{\omega_0}{2\delta} 
$$

In [ ]:
# 1. Experiment

t_int = np.array([])
t_u = np.array([])
t = unp.uarray(t_int, t_u) 
x_int = np.array([])
x_u = np.array([])
x = unp.uarray(x_int, x_u)
x0 = # anfangsamplitude
b = # letzter gemessener zeitwert

T = b/len(t_int)
omega = (2*math.pi)/T 
print(f"Die Schwingungskreisfrequenz beträgt {} rad/s.")

y_1 = np.log(x_int/x0)
res1 = linregress(t_int, y_1)
delta = ufloat(res1.slope, res1.stderr)*(-1) 
y_plot = res1.slope*t_int + res1.intercept
print(f"Die Dämpfungskonstante lässt sich aus der linearen Regression als {delta} kg/s bestimmen.")

plt.figure()
plt.plot(t_int, x_int, fmt="o", label="Messwerte", color="blue")
plt.plot(t_int, y_plot, fmt="o", label="Lineare Regression y=kx+d", color="red")
plt.xlabel("Zeit t [s]")
plt.ylabel("Logarithmische Darstellung der Abnahme der Amplitude")
plt.title("Freie gedämpfte Schwingung")
plt.grid(True)
plt.legend()
plt.show()

omega0 = np.sqrt((omega)**2+(delta)**2)
print(f"Die Eigenfrequenz beträgt {omega0} rad/s.")

# 1.4
A = np.array([]) # auslenkung
T_g = np.array([])/10
f = 1/T_g
omega_2 = (2*math.pi)/f


# modellfunktion

def modell(omega_2, omega0, delta):
    return (omega0**2)/np.sqrt((omega0**2 - omega_2**2)**2 + 4*(delta**2)*omega_2**2)

y_data = A

# fit durchführen
popt, pcov = curve_fit(modell, omega_2, y_data, p0=[1000,1])

omega0_fit, delta_fit = popt
omega0_fit_2 = ufloat(popt[0], np.sqrt(pcov[0,0]))
delta_fit_2 = ufloat(popt[1], np.sqrt(pcov[1,1]))


omega_fit = np.linspace(min(omega_2), max(omega_2), 500)
y_fit = modell(omega_2, omega0_fit, delta_fit)
y_fit_2 = modell(omega_fit, omega0_fit, delta_fit)


# chi2s = np.sum(((U_s_nom - y_fit)/u_U_s)**2) brauchen wir doch nicht und ist auch falsch
SS_res_s = np.sum((A - y_fit)**2)
SS_tot_s = np.sum((A - np.mean(y_data))**2)
R2_s = 1 - (SS_res_s / SS_tot_s) 

# plot

plt.scatter(omega_2, y_data, marker="o", color='red')
plt.plot(omega_fit, y_fit_2, label="Fit", color='blue')
plt.xlabel("Kreisfrequenz omega [rad/s]")
plt.ylabel("Auslenkung")
plt.title("Getriebene gedämpfte Schwingung")
plt.legend()
plt.grid(True)
plt.show()

# HALBWERTSBREITE DANN DELTA_2 

delta_2 = # hier als dämpfungskoeffizient vom 2. experiment definieren
Q_1 = omega0/(2*delta)
Q_2 = omega0/(2*delta_2)
diff = Q_1 - Q_2

print(f"Der Gütefaktor der freien gedämpften Schwingung beträgt {Q_1}, für die erzwungene gedämpfte Schwingung wurde der Gütefaktor {Q_2} berechnet. Zwischen ihnen liegt eine Differenz von {diff}.")





## Erklärung des zweiten Experiments

Im zweiten Experiment werden gekoppelte Schwingungen betrachtet. Zwei Fadenpendel, mit der Kopplungslänge $l$ und der Pendellänge $L$, werden mit einer Aufhängung mit Kopplungsgewicht $G$ verbunden. In dem Kontext der gekoppelten Schwingungen gibt es die gleich- und gegensinnige Schwingung, sowie den Schwebungsfall, welcher hier näher beleuchtet wird. In diesem wird eines der beiden Pendel ausgelenkt, während das andere ruht. Die Energie des ausgelenkten Pendels überträgt sich dabei kontinuierlich auf das anfangs ruhende, welches dann ausgelenkt wird, dieser Prozess wiederholt sich. 
Dabei gibt es in dem System die beiden Eigenschwingungen $\omega_0$ und $\omega_1$, sowie die zu messende Schwebungsfrequenz $\omega_S$. All diese werden wie folgt ermittelt:
$$
\omega_0 = \omega_2 - \omega_S
$$
$$
\omega_1 = \omega_S + \omega_2
$$
$$
\omega_S = \frac{\pi}{T_S}
$$
wobei $\omega_S$ die zu messende Kreisfrequenz ist, mit welcher das Pendel schwingt.
Mit diesen Werten ist nun auch der Kopplungsgrad $K$ zu berechnen:
$$
K = \frac{2\omega_S\omega_2}{\omega_S^2 + \omega_2^2}



In [ ]:
# 2. experiment

l = unp.ufloat( ,1)
L = unp.ufloat( , 1)

T_s = np.array([]) # in s, zeit zwischen zwei stillständen desselben pendels
w_2 = unp.ufloat() # gemessen
w_s = math.pi/T_s
w_0 = w_2 - w_s
w_1 = 2*w_2 - w_0

K = (2*w_s*w_2)/(w_s)**2+(w_2)**2

print(f"Die Eigenfrequenzen der beiden Pendel betragen {w_0} und {w_1} rad/s.")
print(f"Der Kopplungsgrad K wurde als {K} bestimmt.")


## Diskussion

Beim ersten Experiment kann die Messung nicht wirklich genau durchgeführt werden, da exakte Ablesung die menschlichen Fähigkeiten überschreitet. Dementsprechend wurde die Unsicherheit erhöht.